**Installing the Dependencies**

In [ ]:
!pip install --upgrade openai

Import necessary Libraries

In [ ]:
import csv
from openai import OpenAI
import pandas as pd
import re
import requests
from tqdm import tqdm
from sklearn.metrics import classification_report

GITHUB Details - Replace GITHUB ACCESS KEY with your own Personal access token

In [ ]:
github_token = "GITHUB ACCESS KEY"
headers = {"Authorization": f"token {github_token}"}

**Discussion to Issues**

In [ ]:
df = pd.read_csv('DiscussionToIssue.csv')

In [ ]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

Instruction and a function to generate prompt for the LLM

In [ ]:
instructions = """
    Task: Classify the following GitHub Discussion based on the given data:

    'Yes' if an issue was raised from the discussion.
    'No' if it remained just a discussion.
    Definitions:
    Issue (GitHub Issue): A GitHub issue is a formal, trackable unit used to report bugs, request features, or document actionable tasks in a repository. It often includes a structured description, labels, milestones, assignees, and sometimes linked pull requests. Issues are intended for problem-solving and task management within the development workflow.

    Discussion (GitHub Discussion): A GitHub discussion is an open-ended conversation where repository members and contributors can ask questions, share ideas, or engage in broader project-related topics. Discussions are not necessarily tied to immediate tasks or code changes but serve as a collaborative space. However, if a discussion identifies a concrete problem or feature request, it may be converted into an issue.
    """

def getPrompt(msg):
    prompt = f"{instructions}\nData: \"{msg}\"\n\nOutput: Respond with only one of the following categories: Yes, No. Provide no additional text."
    return prompt

Function to get response from GPT-4o

In [ ]:
OPENAI_API_KEY='OPENAIKEY'
client = OpenAI(api_key=OPENAI_API_KEY)

def getGPTResponse(prompt):
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a classifier."},
            {
                "role": "user",
                "content": getPrompt(prompt)
            }
        ],temperature=0
    )

    return completion.choices[0].message.content

GPT-4o

In [ ]:
data = list(df_shuffled['Title'] + ' ' + df_shuffled['Description'] + ' ' + df_shuffled['Comments'])

In [ ]:
df_dict = df_shuffled.to_dict(orient='records')

Generate response for the prompt with Data of Discussion
Title + Description + Comments

In [ ]:
cf_4 = []
for i in tqdm(range(0, len(data))):
    if('gpt-4o-output' not in df_dict[i] or df_dict[i]['gpt-4o-output'] == 'ERROR'):
        prompt = getPrompt(data[i])
        try:
            df_dict[i]['gpt-4o-output'] = getGPTResponse(prompt)
        except:
            cf_4.append(i)
            df_dict[i]['gpt-4o-output'] = 'ERROR'
            pass

100%|██████████| 2218/2218 [21:50<00:00,  1.69it/s]


Function to Save the Output File

In [ ]:
def save_file(csv_file_name,data):
    headers = data[0].keys()

    with open(csv_file_name, 'w', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data)

In [ ]:
save_file('output_DI_All.csv',df_dict)

In [ ]:
df2 = pd.read_csv('output_DI_All.csv')
df2 = df2.dropna(axis=1)

In [ ]:
df2.to_csv('output_DI_All.csv')

Discussion to Issues wit Description Alone

Title + Description

In [ ]:
df = pd.read_csv('DiscussionToIssue.csv')

In [ ]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
data = list(df_shuffled['Title'] + ' ' + df_shuffled['Description'])

In [ ]:
df_dict = df_shuffled.to_dict(orient='records')

Generate response for the Discussion with Title + Description

In [ ]:
cf_4 = []
for i in tqdm(range(0, len(data))):
    if('gpt-4o-output' not in df_dict[i] or df_dict[i]['gpt-4o-output'] == 'ERROR'):
        prompt = getPrompt(data[i])
        try:
            df_dict[i]['gpt-4o-output'] = getGPTResponse(prompt)
        except:
            cf_4.append(i)
            df_dict[i]['gpt-4o-output'] = 'ERROR'
            pass

100%|██████████| 2218/2218 [19:02<00:00,  1.94it/s]


Save the Output

In [ ]:
save_file('output_DI_Description.csv',df_dict)

In [ ]:
df2 = pd.read_csv('output_DI_Description.csv')
df2 = df2.dropna(axis=1)

In [ ]:
df2.to_csv('output_DI_Description.csv')

First Comment Alone

In [ ]:
df = pd.read_csv('DiscussionToIssue.csv')

In [ ]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

Function to extraact the Repository and Discussion Number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Function to get the first comment of the Discussion

In [ ]:
df_shuffled['Comment'] =  None
for index,row in tqdm(df_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df_shuffled.at[index, 'Comment'] = repo_comment

2218it [11:38,  3.18it/s]


In [ ]:
data = list(df_shuffled['Title'] + ' ' + df_shuffled['Description'] + ' ' + str(df_shuffled['Comment']))

In [ ]:
df_dict = df_shuffled.to_dict(orient='records')

Get response from the LLM for the Title + Description + Comment

In [ ]:
cf_4 = []
for i in tqdm(range(0, len(data))):
    if('gpt-4o-output' not in df_dict[i] or df_dict[i]['gpt-4o-output'] == 'ERROR'):
        prompt = getPrompt(data[i])
        try:
            df_dict[i]['gpt-4o-output'] = getGPTResponse(prompt)
        except:
            cf_4.append(i)
            df_dict[i]['gpt-4o-output'] = 'ERROR'
            pass

100%|██████████| 2218/2218 [23:29<00:00,  1.57it/s]


Save the output

In [ ]:
save_file('output_DI_FirstComment.csv',df_dict)

In [ ]:
df2 = pd.read_csv('output_DI_FirstComment.csv')
df2 = df2.dropna(axis=1)

In [ ]:
df2.to_csv('output_DI_FirstComment.csv')

**Issues to Discussion**

Function to generate prompt with relevant instruction

In [ ]:
instructions = """
    Task: Classify the following GitHub Issue based on the given data:

    'Yes' if the issue was converted into a discussion.
    'No' if it remained as an issue.
    Definitions:
    Issue (GitHub Issue): A GitHub issue is a structured, trackable item used to report bugs, request features, or document tasks within a repository. It includes details such as labels, milestones, assignees, and linked pull requests. Issues are intended for actionable problem-solving and development tracking.

    Discussion (GitHub Discussion): A GitHub discussion is a conversational thread used for brainstorming, gathering feedback, or open-ended project-related discussions. Discussions are less structured than issues and do not require immediate resolution. Sometimes, issues that are deemed non-actionable, exploratory, or better suited for community engagement may be converted into discussions.
    """

def getPrompt(msg):
    prompt = f"{instructions}\nData: \"{msg}\"\n\nOutput: Respond with only one of the following categories: Yes, No. Provide no additional text."
    return prompt

Load the dataset

In [ ]:
df1 = pd.read_csv("IssueToDiscussion.csv")

In [ ]:
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
data = list(df1_shuffled['Title'] + ' ' + df1_shuffled['Description'] + ' ' + df1_shuffled['Comments'])

In [ ]:
df_dict = df1_shuffled.to_dict(orient='records')

Generate response with GPT-4o LLM for the data containing Title + Description + Comments

In [ ]:
cf_4 = []
for i in tqdm(range(0, len(data))):
    if('gpt-4o-output' not in df_dict[i] or df_dict[i]['gpt-4o-output'] == 'ERROR'):
        prompt = getPrompt(data[i])
        try:
            df_dict[i]['gpt-4o-output'] = getGPTResponse(prompt)
        except:
            cf_4.append(i)
            df_dict[i]['gpt-4o-output'] = 'ERROR'
            pass

100%|██████████| 2010/2010 [16:18<00:00,  2.05it/s]


Save the Output

In [ ]:
save_file('output_ID_All.csv',df_dict)

In [ ]:
df2 = pd.read_csv('output_ID_All.csv')
df2 = df2.dropna(axis=1)

In [ ]:
df2.to_csv('output_ID_All.csv')

Issue to Discussion with Description Alone

Title  + Description

In [ ]:
df1 = pd.read_csv("IssueToDiscussion.csv")

In [ ]:
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
data = list(df1_shuffled['Description'])

In [ ]:
df_dict = df1_shuffled.to_dict(orient='records')

Generate response with GPT-4o LLM for the data containing Title + Description

In [ ]:
cf_4 = []
for i in tqdm(range(0, len(data))):
    if('gpt-4o-output' not in df_dict[i] or df_dict[i]['gpt-4o-output'] == 'ERROR'):
        prompt = getPrompt(data[i])
        try:
            df_dict[i]['gpt-4o-output'] = getGPTResponse(prompt)
        except:
            cf_4.append(i)
            df_dict[i]['gpt-4o-output'] = 'ERROR'
            pass

 34%|███▎      | 676/2010 [07:14<11:15,  1.97it/s]

Save the Output

In [ ]:
save_file('output_ID_Description.csv',df_dict)

In [ ]:
df2 = pd.read_csv('output_ID_Description.csv')
df2 = df2.dropna(axis=1)

In [ ]:
df2.to_csv('output_ID_Description.csv')

Issues to Discussion with Description and First Comment

Description + First Comment

In [ ]:
df1 = pd.read_csv("IssueToDiscussion.csv")

In [ ]:
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Extract the Repository and Issue Number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the first Comment for each Issue

In [ ]:
df1_shuffled['Comment'] =  None
for index,row in tqdm(df1_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df1_shuffled.at[index, 'Comment'] = repo_comment

2010it [02:33, 13.10it/s]


In [ ]:
data = list(df1_shuffled['Description'] + ' ' + str(df1_shuffled['Comment']))

In [ ]:
df_dict = df1_shuffled.to_dict(orient='records')

Generate response with GPT-4o LLM for the data containing Description + First Comment

In [ ]:
cf_4 = []
for i in tqdm(range(0, len(data))):
    if('gpt-4o-output' not in df_dict[i] or df_dict[i]['gpt-4o-output'] == 'ERROR'):
        prompt = getPrompt(data[i])
        try:
            df_dict[i]['gpt-4o-output'] = getGPTResponse(prompt)
        except:
            cf_4.append(i)
            df_dict[i]['gpt-4o-output'] = 'ERROR'
            pass

100%|██████████| 2010/2010 [15:53<00:00,  2.11it/s]


Save the File

In [ ]:
save_file('output_ID_Description_Comment.csv',df_dict)

In [ ]:
df2 = pd.read_csv('output_ID_Description_Comment.csv')
df2 = df2.dropna(axis=1)

In [ ]:
df2.to_csv('output_ID_Description_Comment.csv')

Issues to Discussion with Comments

In [ ]:
df1 = pd.read_csv("IssueToDiscussion.csv")

In [ ]:
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
data = list(df1_shuffled['Comments'])

In [ ]:
df_dict = df1_shuffled.to_dict(orient='records')

Generate response with GPT-4o LLM for the data containing Comments

In [ ]:
cf_4 = []
for i in tqdm(range(0, len(data))):
    if('gpt-4o-output' not in df_dict[i] or df_dict[i]['gpt-4o-output'] == 'ERROR'):
        prompt = getPrompt(data[i])
        try:
            df_dict[i]['gpt-4o-output'] = getGPTResponse(prompt)
        except:
            cf_4.append(i)
            df_dict[i]['gpt-4o-output'] = 'ERROR'
            pass

100%|██████████| 2010/2010 [16:29<00:00,  2.03it/s]


Save the File

In [ ]:
save_file('output_ID_Comments.csv',df_dict)

In [ ]:
df2 = pd.read_csv('output_ID_Comments.csv')
df2 = df2.dropna(axis=1)

In [ ]:
df2.to_csv('output_ID_Comments.csv')

Function to generate Classification Report for Discussion to Issues

In [ ]:
def generate_classification_report(filename):
    df = pd.read_csv(filename)
    if 'IsIssueRaised' not in df.columns or 'gpt-4o-output' not in df.columns:
        raise ValueError("CSV file must contain 'IsIssueRaised' and 'gpt-4o-output' columns")
    y_true = df['IsIssueRaised']
    y_pred = df['gpt-4o-output']
    if y_true.dtype == 'object' or y_pred.dtype == 'object':
        y_true = y_true.astype(str)
        y_pred = y_pred.astype(str)
    report = classification_report(y_true, y_pred)

    print(report)

Function to generate Classification Report for Issues to Discussion

In [ ]:
def generate_classification_report1(filename):
    df = pd.read_csv(filename)
    if 'ConvertedFromIssue' not in df.columns or 'gpt-4o-output' not in df.columns:
        raise ValueError("CSV file must contain 'IsIssueRaised' and 'gpt-4o-output' columns")
    y_true = df['ConvertedFromIssue']
    y_pred = df['gpt-4o-output']
    if y_true.dtype == 'object' or y_pred.dtype == 'object':
        y_true = y_true.astype(str)
        y_pred = y_pred.astype(str)
    report = classification_report(y_true, y_pred)

    print(report)

Generate Classification Reports

In [ ]:
generate_classification_report('output_DI_All.csv')

              precision    recall  f1-score   support

          No       0.81      0.47      0.59      1000
         Yes       0.68      0.91      0.77      1218

    accuracy                           0.71      2218
   macro avg       0.74      0.69      0.68      2218
weighted avg       0.73      0.71      0.69      2218



In [ ]:
generate_classification_report('output_DI_Description.csv')

              precision    recall  f1-score   support

          No       0.65      0.48      0.55      1000
         Yes       0.65      0.79      0.71      1218

    accuracy                           0.65      2218
   macro avg       0.65      0.63      0.63      2218
weighted avg       0.65      0.65      0.64      2218



In [ ]:
generate_classification_report('output_DI_FirstComment.csv')

              precision    recall  f1-score   support

          No       0.64      0.48      0.55      1000
         Yes       0.65      0.78      0.71      1218

    accuracy                           0.65      2218
   macro avg       0.65      0.63      0.63      2218
weighted avg       0.65      0.65      0.64      2218



In [ ]:
generate_classification_report1('output_ID_All.csv')

              precision    recall  f1-score   support

          No       0.71      0.47      0.56       997
         Yes       0.61      0.81      0.70      1013

    accuracy                           0.64      2010
   macro avg       0.66      0.64      0.63      2010
weighted avg       0.66      0.64      0.63      2010



In [ ]:
generate_classification_report1('output_ID_Comments.csv')

              precision    recall  f1-score   support

          No       0.53      0.72      0.61       997
         Yes       0.57      0.37      0.45      1013

    accuracy                           0.54      2010
   macro avg       0.55      0.54      0.53      2010
weighted avg       0.55      0.54      0.53      2010



In [ ]:
generate_classification_report1('output_ID_Description.csv')

              precision    recall  f1-score   support

          No       0.70      0.52      0.60       997
         Yes       0.62      0.78      0.69      1013

    accuracy                           0.65      2010
   macro avg       0.66      0.65      0.65      2010
weighted avg       0.66      0.65      0.65      2010



In [ ]:
generate_classification_report1('output_ID_Description_Comment.csv')

              precision    recall  f1-score   support

          No       0.68      0.51      0.58       997
         Yes       0.61      0.77      0.68      1013

    accuracy                           0.64      2010
   macro avg       0.65      0.64      0.63      2010
weighted avg       0.65      0.64      0.63      2010

